In [6]:
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader 

In [3]:
#!pip install tensorboard

Note: you may need to restart the kernel to use updated packages.


In [5]:
writer = SummaryWriter()

In [14]:
NUM_WORKERS = os.cpu_count()

def create_dataloaders(train_dir, test_dir, transform, batch_size, num_workers):
    train_data = datasets.ImageFolder(train_dir, transform = transform)
    test_data = datasets.ImageFolder(test_dir, transform = transform)

    class_names = train_data.classes

    train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle = True, num_workers=NUM_WORKERS, pin_memory=True)
    test_dataloader = DataLoader(test_data, batch_size, False, num_workers=NUM_WORKERS, pin_memory=True)

    return train_dataloader, test_dataloader, class_names

In [8]:
class DesertClassifier(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)  # default stride value is same as kernel_size
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)  
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features= hidden_units * 16 * 16,
                      out_features=output_shape)
        )

    def forward(self, x):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

In [10]:
def train_step(model, dataloader, loss_fn, optimizer):
    model.train()

    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        # forward pass
        y_pred = model(X)

        # calculate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # optimizer zero grad
        optimizer.zero_grad()

        # loss backward
        loss.backward()

        # optimizer step
        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum() / len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)

    return train_loss, train_acc



In [11]:
def test_step(model, dataloader, loss_fn):
    model.eval()

    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):

            # forward pass
            test_pred_logits = model(X)

            # calculate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # calculate acc
            test_pred_labels = test_pred_logits.argmax(dim = 1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [12]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5,
          experiment_name="experiment"):
    # 2. Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
               }

    # 3. Loop through training and testing steps for a number of epochs
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn)

        # 4. Print out what's happening
        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)
        
        writer = SummaryWriter(log_dir=f"runs/{experiment_name}")
        # Add loss results to SummaryWriter
        writer.add_scalars(main_tag="Loss", 
                           tag_scalar_dict={"train_loss": train_loss,
                                            "test_loss": test_loss},
                           global_step=epoch)

        # Add accuracy results to SummaryWriter
        writer.add_scalars(main_tag="Accuracy", 
                           tag_scalar_dict={"train_acc": train_acc,
                                            "test_acc": test_acc}, 
                           global_step=epoch)
        
        # Track the PyTorch model architecture
        writer.add_graph(model=model, 
                         # Pass in an example input
                         input_to_model=torch.randn(32, 3, 64, 64))
    
    # Close the writer
    writer.close()

    # 6. Return the filled results at the end of the epochs
    return results

In [16]:
    NUM_EPOCHS = 2
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001

    # Setup directories
    train_dir = "data/desert101/train"
    test_dir = "data/desert101/test"

    #device = "cuda" if torch.cuda.is_available() else "cpu"

    # Create transforms
    data_transform = transforms.Compose([
      transforms.Resize((64, 64)),
      transforms.ToTensor()
    ])

    # Create DataLoaders with help from data_setup.py
    train_dataloader, test_dataloader, class_names = create_dataloaders(
        train_dir=train_dir,
        test_dir=test_dir,
        transform=data_transform,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS
    )

    # Create model with help from model_builder.py
    model = DesertClassifier(
        input_shape=3,
        hidden_units=10,
        output_shape=len(class_names)
    )

    model2 = DesertClassifier(
        input_shape=3,
        hidden_units=32,
        output_shape=len(class_names)
    )

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [18]:
train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer=torch.optim.Adam(model.parameters(), lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    experiment_name="hidden_units_10"
)

Epoch: 1 | train_loss: 1.3962 | train_acc: 0.2152 | test_loss: 1.3894 | test_acc: 0.1979
Epoch: 2 | train_loss: 1.3906 | train_acc: 0.2402 | test_loss: 1.3927 | test_acc: 0.1979


{'train_loss': [1.3962297320365906, 1.3905913472175597],
 'train_acc': [0.21517856419086456, 0.24017855525016785],
 'test_loss': [1.3893773953119914, 1.392664869626363],
 'test_acc': [0.19791666666666666, 0.19791666666666666]}

In [19]:
train(
    model=model2,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer=torch.optim.Adam(model2.parameters(), lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    experiment_name="hidden_units_32"
)

Epoch: 1 | train_loss: 1.3927 | train_acc: 0.1996 | test_loss: 1.3879 | test_acc: 0.2083
Epoch: 2 | train_loss: 1.3798 | train_acc: 0.2696 | test_loss: 1.3732 | test_acc: 0.2396


{'train_loss': [1.392740833759308, 1.3797622561454772],
 'train_acc': [0.19955357909202576, 0.2696428596973419],
 'test_loss': [1.3878698348999023, 1.3731865485509236],
 'test_acc': [0.20833333333333334, 0.23958333333333334]}

In [20]:
%load_ext tensorboard
%tensorboard --logdir runs # run session with the "runs/" directory (runs directory is created automatically)